In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as dsets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# 1. Environment Setup and Hyperparameter Definition
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 128
lr = 0.0002
num_epochs = 30
z_dim = 100  # Dimension of the latent space

# 2. Loading and Preprocessing the Dataset (MNIST)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize pixel values to the range [-1, 1] (suitable for Tanh)
])
dataset = dsets.MNIST(root='./data', train=True, transform=transform, download=True)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 3. Implementing the Generator Model
class Generator(nn.Module):
    def __init__(self, z_dim=100, img_shape=(1, 28, 28)):
        super(Generator, self).__init__()
        self.img_shape = img_shape
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, int(np.prod(img_shape))),
            nn.Tanh()  # Adjust output to the range [-1, 1]
        )

    def forward(self, z):
        img = self.model(z)
        # Reshape the output vector into image form
        img = img.view(z.size(0), *self.img_shape)
        return img

# 4. Implementing the Discriminator Model
class Discriminator(nn.Module):
    def __init__(self, img_shape=(1, 28, 28)):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(int(np.prod(img_shape)), 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()  # Output a value between 0 and 1 indicating whether the image is real or fake
        )

    def forward(self, img):
        # Flatten the image into a 1D vector
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity

# 5. Creating the Models and Allocating them to the Device
generator = Generator(z_dim=z_dim).to(device)
discriminator = Discriminator().to(device)

# 6. Setting Up the Loss Function and Optimizers
adversarial_loss = nn.BCELoss()  # Binary classification loss function (real / fake)
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

# 7. Weight Initialization (Commonly used in GANs: normal distribution with mean 0 and std 0.02)
def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find('Linear') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)

generator.apply(weights_init_normal)
discriminator.apply(weights_init_normal)

# 8. Training Process
G_losses = []  # Record Generator loss
D_losses = []  # Record Discriminator loss

for epoch in range(num_epochs):
    for i, (imgs, _) in enumerate(data_loader):
        batch_size_i = imgs.size(0)

        # Define labels for real and fake (real: 1, fake: 0)
        valid = torch.ones((batch_size_i, 1), device=device)
        fake = torch.zeros((batch_size_i, 1), device=device)

        real_imgs = imgs.to(device)

        # ----- Train Generator -----
        optimizer_G.zero_grad()
        # Sample random noise from the latent space
        z = torch.randn(batch_size_i, z_dim, device=device)
        gen_imgs = generator(z)
        # The goal of the Generator is to make the generated images be classified as real by the Discriminator
        g_loss = adversarial_loss(discriminator(gen_imgs), valid)
        g_loss.backward()
        optimizer_G.step()

        # ----- Train Discriminator -----
        optimizer_D.zero_grad()
        # Loss for real images
        real_loss = adversarial_loss(discriminator(real_imgs), valid)
        # Loss for fake images (using detach to prevent Generator's gradients from being updated)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake) #  to avoid backpropagation for generator's grandients
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        # Record losses
        G_losses.append(g_loss.item())
        D_losses.append(d_loss.item())

        if i % 100 == 0:
            print(f"[Epoch {epoch+1}/{num_epochs}] [Batch {i}/{len(data_loader)}] [D loss: {d_loss.item():.4f}] [G loss: {g_loss.item():.4f}]")

    # Visualize generated images at the end of each epoch
    generator.eval()  # Set to evaluation mode
    with torch.no_grad():
        # Using fixed noise makes it easier to observe changes across epochs
        fixed_noise = torch.randn(25, z_dim, device=device)
        gen_imgs = generator(fixed_noise)
    generator.train()

    # Convert image scale from [-1,1] to [0,1]
    gen_imgs = (gen_imgs + 1) / 2
    grid = torchvision.utils.make_grid(gen_imgs, nrow=5, padding=2, normalize=False)
    np_grid = grid.cpu().numpy().transpose((1, 2, 0))

    plt.figure(figsize=(5,5))
    plt.imshow(np_grid)
    plt.title(f"Epoch {epoch+1}")
    plt.axis('off')
    plt.show()

# Visualize loss curves after training
plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses, label="Generator Loss")
plt.plot(D_losses, label="Discriminator Loss")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()


Output hidden; open in https://colab.research.google.com to view.